## Load modules

In [1]:
!module load scipy-stack/2023b
!module list

Lmod Warning: 
-------------------------------------------------------------------------------
The following dependent module(s) are not currently loaded: ipykernel/2026a
(required by: ipython-kernel/3.11)
-------------------------------------------------------------------------------




Currently Loaded Modules:
  1) CCconfig                 16) calibre/8.6.0
  2) gentoo/2023         (S)  17) libreqda/1.1.0
  3) gcccore/.12.3       (H)  18) flexiblascore/.3.3.1         (H)
  4) gcc/12.3            (t)  19) java/17 -> java/17.0.6       (t)
  5) hwloc/2.9.1              20) r/4.4.0                      (t)
  6) ucx/1.14.1               21) rstudio-server/4.4           (t)
  7) libfabric/1.18.0         22) python/3.11 -> python/3.11.5 (t)
  8) pmix/4.2.4               23) ipython-kernel/3.11          (S)
  9) ucc/1.2.0                24) openrefine/3.9.3
 10) openmpi/4.1.5       (m)  25) mlflow/3.8.1
 11) flexiblas/3.3.1          26) tensorboard/2.20.0
 12) blis/0.9.0               27) 

In [2]:
import numpy as np
import pandas as pd
import math as math
import matplotlib.pyplot as plt
import subprocess
import os
import scipy.stats as stats
import seaborn as sns
from collections import Counter
from tqdm import tqdm
import glob
from matplotlib.lines import Line2D

In [3]:
input_datasheet = pd.read_csv('../ChrR_datasheet.csv', sep=',', index_col=0)
df = pd.read_csv('../2.Filtering_and_Statistics/statistics_df_with_log2fc_and_fdr.csv', sep=',', index_col=0)
df

,Alias,Gene,Barcode,Position,SC5314_TP0_YPD_1,SC5314_TP0_YPD_2,SC5314_TP0_YPD_3,SC5314_TP0_YPD_4,SC5314_TP3_YPD_1,SC5314_TP3_YPD_2,...,freq_SC5314_TP3_High_FLZ_4,log2_fold_change_ypd,log2_fold_change_low_flz,log2_fold_change_high_flz,p_value_ypd,p_value_low_flz,p_value_high_flz,adj_p_value_ypd,adj_p_value_low_flz,adj_p_value_high_flz
Number,,,,,,,,,,,,,,,,,,,,,
2,CR07390CA_276_revcom,CR07390C_A,GAGTATAGTGATCCATGTGC,1605740.0,2017.0,1006.0,956.0,2949.0,344.0,811.0,...,0.000087,-1.763045,-1.374599,-1.698364,0.020286,0.062619,0.048008,0.105319,0.165318,0.150390
3,CR07390CA_239_revcom,CR07390C_A,GCTATAACGTTACTAGTAGT,1605777.0,88867.0,86080.0,83323.0,116333.0,94225.0,81253.0,...,0.007555,-0.063840,-0.246713,-0.363103,0.043567,0.000031,0.000792,0.138629,0.010235,0.023756
5,CR07390CA_212_revcom,CR07390C_A,TCCATCATCATTATTACAAT,1605804.0,1182.0,862.0,952.0,1604.0,1415.0,943.0,...,0.000091,0.141380,-0.284233,0.289939,0.437345,0.113461,0.518573,0.586304,0.234853,0.664216
6,CR07390CA_70,CR07390C_A,CTCTTTTGCTCTCATTTTTA,1605924.0,1006.0,1401.0,1421.0,1415.0,1658.0,2027.0,...,0.000149,0.497221,0.306597,-0.647516,0.034235,0.151721,0.090439,0.127575,0.285813,0.220006
8,CR10440WA_257,CR10440W_A,AAGAATATTGTGTTGGTTCC,2222738.0,905.0,765.0,806.0,1251.0,1209.0,462.0,...,0.000119,-0.088208,-0.789768,0.051567,0.777664,0.098734,0.880446,0.855045,0.214988,0.930291
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5695,non-targeting55,non-targeting55,AGTGTCCCGTCATGCTCCAG,NaN,2536.0,3181.0,2740.0,4228.0,3344.0,3844.0,...,0.000383,0.132038,0.413062,0.089142,0.338498,0.010153,0.742491,0.497554,0.067718,0.839223
5696,non-targeting56,non-targeting56,TCTTTCGGGAGCGCCTGATT,NaN,3300.0,3063.0,2857.0,4753.0,2746.0,2848.0,...,0.000319,-0.335703,-0.288746,-0.289210,0.010873,0.248795,0.485359,0.092559,0.399547,0.638237
5697,non-targeting57,non-targeting57,TCCCCAAGGCTGTCCCCCAA,NaN,1437.0,926.0,1189.0,1853.0,1421.0,672.0,...,0.000108,-0.335985,-0.427021,-0.787231,0.261365,0.300391,0.020036,0.412575,0.456651,0.091752


## Plotting

Highlight the FLZ-unique hits (ie. hits are only considered if they were NOT also a hit in the YPD-only condition) and add the non-targeting sgRNA confidence intervals.

In [7]:
#Directories
output_dir = "./volcano_plots"
os.makedirs(output_dir, exist_ok=True)

#Thresholds. Absolute logfc thresholds are optional- here we will use the thresholds based on the non-targeting sgRNA confidence interval, so this is left at 0
fdr_thresh = 0.05
logfc_thresh = 0

#List of comparisons with column names in the .csv: (log2FC column, adjusted p-value column)
comparisons = [
    ("log2_fold_change_ypd",      "adj_p_value_ypd"),
    ("log2_fold_change_low_flz",  "adj_p_value_low_flz"),
    ("log2_fold_change_high_flz", "adj_p_value_high_flz"),
]

#Columns
high_flz_logfc_col = "log2_fold_change_high_flz"
high_flz_fdr_col   = "adj_p_value_high_flz"
low_flz_logfc_col  = "log2_fold_change_low_flz"
low_flz_fdr_col    = "adj_p_value_low_flz"
ypd_logfc_col      = "log2_fold_change_ypd"
ypd_fdr_col        = "adj_p_value_ypd"

do_flz_only_highlight = True

#Filter non-targeting guides correctly (safer to avoid NaNs)
non_targeting_df = df[df['Alias'].str.startswith('non-targeting', na=False)]

#Compute means and 95% CI for each comparison (based on non-targeting logFC distribution)
#This is basically a normal distribution-based CI on the non-targeting logFC values (ie. contains ~95% of individual non-targeting sgRNA log2FC values)
ci_dict = {}
for logfc_col, _ in comparisons:
    nt_vals = non_targeting_df[logfc_col].dropna()
    n_nt = nt_vals.shape[0]
    mean_fc = nt_vals.mean()
    std_fc = nt_vals.std()
    ci_lower = mean_fc - 1.96 * std_fc
    ci_upper = mean_fc + 1.96 * std_fc
    ci_dict[logfc_col] = (mean_fc, ci_lower, ci_upper)
    print(f"{logfc_col}: n_non_targeting={n_nt}, mean={mean_fc:.4f}, 95% CI = [{ci_lower:.4f}, {ci_upper:.4f}]")

#Build YPD direction dictionary
ypd_mean, ypd_ci_lower, ypd_ci_upper = ci_dict[ypd_logfc_col]
ypd_outside_ci = (df[ypd_logfc_col] < ypd_ci_lower) | (df[ypd_logfc_col] > ypd_ci_upper)

ypd_sig_outside_ci = (
    (df[ypd_fdr_col] < fdr_thresh) &
    (df[ypd_logfc_col].abs() > logfc_thresh) &
    ypd_outside_ci
)

#Build a lookup of YPD-significant hits and their direction (+1 enriched, -1 depleted) so that FLZ hits in the same "direction" as YPD can be flagged as shared rather than FLZ-unique
tmp = df.loc[ypd_sig_outside_ci, ['Alias', ypd_logfc_col]].dropna()
ypd_direction = dict(zip(tmp['Alias'], np.sign(tmp[ypd_logfc_col])))

#Legend handles
legend_handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor='red', markersize=5, label='FLZ-Unique Significant'),
    Line2D([0], [0], marker='o', color='none', markerfacecolor='black',  markersize=5, label='Significant'),
    Line2D([0], [0], marker='o', color='none', markerfacecolor='grey', markersize=5, label='Not significant')
]

#Plotting
for logfc_col, fdr_col in comparisons:
    df['minus_log10_FDR'] = -np.log10(df[fdr_col].replace(0, np.nextafter(0, 1)))

    #CI for Tthe current plot's logFC distribution (non-targeting)
    mean_fc, ci_lower, ci_upper = ci_dict[logfc_col]

    #Treat anything inside CI as NOT significant
    outside_ci = (df[logfc_col] < ci_lower) | (df[logfc_col] > ci_upper)

    #"Significant" for the current plot means they pass the FDR/logFC thresholds AND is outside the NTC CI
    sig_this = (df[fdr_col] < fdr_thresh) & (df[logfc_col].abs() > logfc_thresh) & outside_ci
    df['significant'] = sig_this

    #FLZ-unique highlighting ONLY on the corresponding FLZ plot, and ONLY exclude YPD hits if they score in the SAME DIRECTION (+ or -)
    is_flz_plot = logfc_col in {low_flz_logfc_col, high_flz_logfc_col}

    if do_flz_only_highlight and is_flz_plot:
        #Same-direction YPD exclusion mask
        ypd_dir_series = df['Alias'].map(ypd_direction)  # NaN for non-YPD hits
        same_direction_as_ypd = (ypd_dir_series.notna()) & (np.sign(df[logfc_col]) == ypd_dir_series)
        flz_unique = sig_this & (~same_direction_as_ypd)
    else:
        flz_unique = np.zeros(len(df), dtype=bool)

    #Colours: red (FLZ-unique) > black (significant) > grey
    colors = np.where(flz_unique, 'red', np.where(sig_this, 'black', 'grey'))
    plt.figure(figsize=(5, 5))
    plt.scatter(
        df[logfc_col], df['minus_log10_FDR'],
        c=colors,
        s=13, edgecolors='none'
    )

    #Fixed y-axis limits
    y_min = 0
    y_max = 3
    plt.ylim(y_min, y_max)

    #FDR threshold line
    plt.axhline(-np.log10(fdr_thresh), color='black', ls='--', lw=1)

    #Add shaded confidence interval for non-targeting guides
    plt.fill_betweenx([y_min, y_max], ci_lower, ci_upper, color='gray', alpha=0.2, edgecolor='none')
    plt.xlabel('log2 Fold Change', fontsize=10, fontweight="bold")
    plt.ylabel('-log10 FDR', fontsize=10, fontweight="bold")
    ax = plt.gca()
    ax.tick_params(axis='both', labelsize=10)
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')

    #Add legend only on the FLZ plots (wouldn't make sense on the YPD plots)
    if is_flz_plot and do_flz_only_highlight:
        leg = ax.legend(
            handles=legend_handles,
            frameon=False,
            fontsize=8,
            loc='upper left',
            handletextpad=0.4,
            borderpad=0.2,
            labelspacing=0.3,
        )
        for text in leg.get_texts():
            text.set_fontweight('bold')

    #Add FLZ condition label in top-right
    if logfc_col == high_flz_logfc_col:
        ax.text(
            0.98, 0.98,
            '64μg/mL FLZ',
            transform=ax.transAxes,
            ha='right',
            va='top',
            fontsize=10,
            fontweight='bold'
        )

    elif logfc_col == low_flz_logfc_col:
        ax.text(
            0.98, 0.98,
            '1μg/mL FLZ',
            transform=ax.transAxes,
            ha='right',
            va='top',
            fontsize=10,
            fontweight='bold'
        )

    plt.tight_layout()
    plot_path = os.path.join(output_dir, f"volcano_{logfc_col}_with_CI.png")
    plt.savefig(plot_path, dpi=300)
    plt.close()

log2_fold_change_ypd: n_non_targeting=25, mean=-0.3510, 95% CI = [-1.4871, 0.7852]
log2_fold_change_low_flz: n_non_targeting=25, mean=-0.3038, 95% CI = [-1.5968, 0.9892]
log2_fold_change_high_flz: n_non_targeting=25, mean=-0.2200, 95% CI = [-1.4210, 0.9810]


In [8]:
#Summary table of the sgRNAs enriched/depleted beyond the non-targeting CI in FLZ conditions
records = []

for logfc_col, fdr_col in [
    (low_flz_logfc_col,  "adj_p_value_low_flz"),
    (high_flz_logfc_col, "adj_p_value_high_flz"),
]:
    mean_fc, ci_lower, ci_upper = ci_dict[logfc_col]
    outside_ci = (df[logfc_col] < ci_lower) | (df[logfc_col] > ci_upper)
    sig_mask   = (
        (df[fdr_col] < fdr_thresh) &
        (df[logfc_col].abs() > logfc_thresh) &
        outside_ci
    )

    #Direction split
    enriched_mask  = sig_mask & (df[logfc_col] > 0)
    depleted_mask  = sig_mask & (df[logfc_col] < 0)

    #FLZ-unique: same-direction YPD exclusion
    ypd_dir_series     = df['Alias'].map(ypd_direction)
    same_dir_as_ypd    = ypd_dir_series.notna() & (np.sign(df[logfc_col]) == ypd_dir_series)
    flz_unique_mask    = sig_mask & ~same_dir_as_ypd
    flz_unique_enr  = flz_unique_mask & (df[logfc_col] > 0)
    flz_unique_dep  = flz_unique_mask & (df[logfc_col] < 0)
    shared_enr      = enriched_mask   & same_dir_as_ypd
    shared_dep      = depleted_mask   & same_dir_as_ypd

    label = "Low FLZ" if logfc_col == low_flz_logfc_col else "High FLZ"

    records.append({
        "Condition":              label,
        "Total significant":      int(sig_mask.sum()),
        "Enriched (total)":       int(enriched_mask.sum()),
        "Depleted (total)":       int(depleted_mask.sum()),
        "FLZ-unique enriched":    int(flz_unique_enr.sum()),
        "FLZ-unique depleted":    int(flz_unique_dep.sum()),
        "Shared enriched (YPD)":  int(shared_enr.sum()),
        "Shared depleted (YPD)":  int(shared_dep.sum()),
        "CI lower":               round(ci_lower, 4),
        "CI upper":               round(ci_upper, 4),
    })

summary_df = pd.DataFrame(records)

#Save CSV and print table
csv_path = os.path.join(output_dir, "flz_sgrna_summary.csv")
summary_df.to_csv(csv_path, index=False)
print(summary_df.to_string(index=False))

Condition  Total significant  Enriched (total)  Depleted (total)  FLZ-unique enriched  FLZ-unique depleted  Shared enriched (YPD)  Shared depleted (YPD)  CI lower  CI upper
  Low FLZ                 68                27                41                   27                   40                      0                      1   -1.5968    0.9892
 High FLZ                129                76                53                   76                   51                      0                      2   -1.4210    0.9810
